# 3. Baselines, Anomalies & the Hunting Maturity Model

The previous notebooks used **signatures** (known-bad IPs, known attack tools, fixed thresholds). Real attackers know your signatures — so mature hunting also looks for **anomalies**: activity that deviates from *this environment's normal baseline*.

This notebook teaches:

1. How to build a **baseline** from historical data.
2. How to detect **anomalies** (off-hours sign-ins, unusual volumes).
3. The **hunting maturity model** — from ad-hoc to automated.
4. A small **MITRE ATT&CK coverage** view over the hunts you've written.

We'll inject a small time-spread dataset (7 days) just for this notebook so the baseline math is meaningful. This does not affect the other labs.


In [ ]:
import httpx, json, random, statistics
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone

SIEM = 'http://localhost:8000'

def _iso(dt):
    return dt.replace(tzinfo=None).isoformat()

# ---- Inject a 7-day baseline for user 'bob', plus one anomaly window ----
random.seed(42)
now = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
entries = []

# Normal pattern: bob signs in ~5 times/day during business hours (9-17 UTC)
for days_ago in range(7, 0, -1):
    day = now - timedelta(days=days_ago)
    for _ in range(5):
        hour = random.randint(9, 17)
        minute = random.randint(0, 59)
        ts = day.replace(hour=hour, minute=minute, second=0)
        entries.append({
            'table_name': 'SigninLogs',
            'timestamp': _iso(ts),
            'data': {
                'UserPrincipalName': 'bob@contoso.com',
                'IPAddress': '10.0.1.11',
                'Location': 'Seattle',
                'ResultType': 'Success',
                'AppDisplayName': 'Outlook',
            },
        })

# Anomaly: three sign-ins from a suspicious IP in the *recent* (last-hour) window.
# This is what the hunt should flag: bob's account used from Moscow NOW.
for _ in range(3):
    ts = now - timedelta(minutes=random.randint(1, 55))
    entries.append({
        'table_name': 'SigninLogs',
        'timestamp': _iso(ts),
        'data': {
            'UserPrincipalName': 'bob@contoso.com',
            'IPAddress': '185.220.101.42',
            'Location': 'Moscow',
            'ResultType': 'Success',
            'AppDisplayName': 'Azure Portal',
        },
    })

r = httpx.post(f'{SIEM}/ingest/batch', json={'entries': entries})
print(f'Injected {len(entries)} time-spread sign-ins for baseline analysis: {r.json()}')


## Step 1 — Build a baseline (hour-of-day profile)

We want to answer: *what does a normal day look like for this user?*  In real KQL, this is one query:

```kusto
SigninLogs
| where UserPrincipalName == "bob@contoso.com"
| where TimeGenerated between (ago(7d) .. ago(1h))
| summarize count() by Hour = bin(TimeGenerated % 1d, 1h)
```

Our SIEM API doesn't compute `bin()` server-side, so we group in Python — but the *concept* is identical.


In [ ]:
def query(table, filter=None, limit=1000):
    return httpx.post(f'{SIEM}/query', json={
        'table_name': table, 'filter': filter, 'limit': limit
    }).json()['results']

signins = query('SigninLogs', filter={'UserPrincipalName': 'bob@contoso.com'})
print(f'Total sign-ins for bob: {len(signins)}\n')

# Split: baseline (older than 1h) vs recent (last 1h)
cutoff = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(hours=1)
def _parse(ts):
    return datetime.fromisoformat(ts.replace('Z',''))

baseline = [s for s in signins if _parse(s['timestamp']) < cutoff]
recent   = [s for s in signins if _parse(s['timestamp']) >= cutoff]

baseline_hours = Counter(_parse(s['timestamp']).hour for s in baseline)
print('Hour-of-day baseline (last 7d, excluding current hour):')
for h in range(24):
    bar = '█' * baseline_hours.get(h, 0)
    marker = '  ← business hours' if 9 <= h <= 17 else ''
    print(f'  {h:02d}:00  {baseline_hours.get(h,0):>2} {bar}{marker}')


## Step 2 — Detect anomalies against the baseline

Two simple, widely-used techniques:

| Technique | What it catches |
|-----------|-----------------|
| **Off-hours activity** | An event at an hour where the baseline is ~0. Cheap and effective. |
| **Statistical outlier (mean + N·σ)** | A daily/hourly count that is several standard deviations above the mean. Used by UEBA tools. |

Real KQL examples:

```kusto
// 1. Off-hours
SigninLogs | extend Hour = hourofday(TimeGenerated)
| where Hour !between (9 .. 17)

// 2. Statistical outlier using series_decompose_anomalies
SigninLogs
| make-series Count=count() on TimeGenerated step 1h by UserPrincipalName
| extend anomalies = series_decompose_anomalies(Count)
```


In [ ]:
# --- Anomaly 1: off-hours activity for the recent window ---
print('=== Anomaly check: off-hours sign-ins in the last hour ===\n')
BUSINESS = range(9, 18)  # 9..17
off_hours = [s for s in recent if _parse(s['timestamp']).hour not in BUSINESS]
for s in off_hours:
    hour = _parse(s['timestamp']).hour
    baseline_count = baseline_hours.get(hour, 0)
    print(f'  🔴 {s["timestamp"][:19]}  user={s["UserPrincipalName"]}')
    print(f'      hour={hour:02d}:00  baseline={baseline_count} signins at this hour → ANOMALY')
    print(f'      IP={s["IPAddress"]}  Location={s["Location"]}')

if not off_hours:
    print('  No off-hours anomalies in the last hour.')


In [ ]:
# --- Anomaly 2: statistical outlier on per-day counts ---
print('\n=== Anomaly check: daily volume outlier (mean ± 2σ) ===\n')

per_day = Counter(_parse(s['timestamp']).date() for s in signins)
# Drop today (partial day) from the baseline sample
today = datetime.now(timezone.utc).date()
sample = [c for d, c in per_day.items() if d != today]
today_count = per_day.get(today, 0)

if len(sample) >= 2:
    mean = statistics.mean(sample)
    stdev = statistics.pstdev(sample) or 1.0  # guard against 0
    upper = mean + 2 * stdev
    lower = max(0, mean - 2 * stdev)
    print(f'  Baseline daily count: mean={mean:.1f}  stdev={stdev:.1f}  → expected [{lower:.1f}, {upper:.1f}]')
    print(f'  Today so far: {today_count} sign-ins')
    if today_count > upper:
        print(f'  🔴 ANOMALY: today\'s count exceeds +2σ band.')
    elif today_count < lower:
        print(f'  🟡 Lower-than-normal volume (could indicate an outage).')
    else:
        print(f'  ✅ Today is within normal range.')
else:
    print('  Not enough baseline days yet.')


## Step 3 — The hunting maturity model

Use this ladder to self-assess a hunting program:

| Level | Description | Example |
|-------|-------------|---------|
| 0 — **Initial** | No hunts. Reactive to alerts only. | "We look at Defender alerts when they fire." |
| 1 — **Ad-hoc hunts** | Manual, opportunistic, undocumented. | Someone greps logs after reading a news article. |
| 2 — **Hypothesis-driven** | Written hypotheses, repeatable queries, tied to MITRE ATT&CK. | "Hunt for T1059.001 PowerShell execution weekly." |
| 3 — **Data-driven / baseline** | Uses statistics on your own data: anomalies, UEBA, rare-in-environment. | Today's notebook. |
| 4 — **Automated** | Successful hunts become scheduled detections + playbooks. | Hunt 5 in notebook 1 → rule in notebook 2. |

The goal isn't to stay at level 4 forever — it's to constantly *promote* hunts up the ladder: ad-hoc → hypothesis → baseline → automated.


In [ ]:
# --- MITRE ATT&CK coverage view for this lab's hunts ---
hunts = [
    # (hunt_id,            tactic,            technique,  notebook)
    ('H-1 Failed signins', 'CredentialAccess','T1110',   '01'),
    ('H-2 Suspicious loc', 'InitialAccess',   'T1078',   '01'),
    ('H-3 Rare processes', 'Execution',       'T1059',   '01'),
    ('H-4 Bad-IP outbound','Exfiltration',    'T1041',   '01'),
    ('H-5 Brute success',  'CredentialAccess','T1110.001','01'),
    ('H2-1 Brute force',   'CredentialAccess','T1110',   '02'),
    ('H2-2 Lateral move',  'LateralMovement', 'T1021',   '02'),
    ('H2-3 Exfiltration',  'Exfiltration',    'T1041',   '02'),
    ('H2-4 TI watchlist',  '(cross-cutting)', 'IOC-match','02'),
    ('H3-1 Off-hours',     'InitialAccess',   'T1078',   '03'),
    ('H3-2 Volume anomaly','Discovery/UEBA',  'T1078',   '03'),
]

print(f'{"Hunt":<22} {"MITRE Tactic":<20} {"Technique":<12} Notebook')
print('-' * 65)
for h, tac, tech, nb in hunts:
    print(f'{h:<22} {tac:<20} {tech:<12} {nb}')

tactics = Counter(t for _, t, _, _ in hunts)
print('\nCoverage by tactic:')
for tac, c in tactics.most_common():
    bar = '█' * c
    print(f'  {tac:<20} {c} {bar}')


## You've completed all SC-200 labs!

### What you built and practiced

1. A **working mini-SIEM** with log ingestion, query engine, analytics rules, watchlists, and playbooks.
2. **Multi-stage attack investigation** across 4 data sources.
3. **Incident response workflows** — triage, investigate, contain, remediate, close.
4. **Threat hunting** at all four maturity levels — ad-hoc, hypothesis-driven, baseline/anomaly, and automated.
5. **MITRE ATT&CK mapping** for every hunt you wrote.

### Next steps

1. Take the [SC-200 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/security-operations-analyst/practice/assessment?assessment-type=practice&assessmentId=59&practice-assessment-type=certification)
2. Practice KQL at [detective.kusto.io](https://detective.kusto.io)
3. Explore real hunting queries: [Azure-Sentinel GitHub repo](https://github.com/Azure/Azure-Sentinel/tree/master/Hunting%20Queries)
4. Read the Microsoft Learn paths listed in the main README
